# Advanced RAG 실습 — 명지대학교 규정 검색 RAG 고도화

> **이 실습에서 할 일 (한 줄 요약)**
> 3장에서 만든 기본 RAG에 **검색 전 절차(데이터 정비·인덱스 전략)** 와 **검색 후 절차(하이브리드·리랭크·압축·질의 변환)** 를 하나씩 붙여 보고, 같은 문제지로 **점수가 얼마나 좋아지는지** 확인합니다.

### 전체 흐름 (강의자료 '5. Advanced RAG' 순서)

```
[Part A. 원천 데이터 정비]   ① 불필요 데이터(부칙) 제거  ② LLM 정련  ③ 중복 데이터 그룹화
[Part B. Retrieval 전략]     기본 청킹 vs 계층 인덱스 · Parent-Child · Sentence Window · HyQE · HyDE · Atomic
[Part C. Search 전략]        Hybrid Search(BM25+벡터) · Sparse/Dense · Metadata Filtering(Pre/Post)
[Part D. 검색 후 절차]       ReRank · 프롬프트 압축 · StepBack · 서브쿼리
[Part E. Collection 분리]    편(Part)별 컬렉션 + 라우팅 · 샤딩 설정
[종합]                       기본 RAG vs Advanced RAG 점수 비교
```

### 꼭 알아야 할 용어
| 용어 | 쉬운 뜻 |
|---|---|
| Pre-Retrieval (검색 전 절차) | 검색하기 **전에** 데이터·질문을 다듬어 검색 품질을 높이는 일 |
| Post-Retrieval (검색 후 절차) | 검색한 **뒤에** 결과를 골라내고 정리해서 LLM에 잘 넘기는 일 |
| BM25 (Sparse) | **단어가 똑같이** 들어 있는지로 찾는 키워드 검색 |
| Dense (벡터) | **뜻이 비슷한지**로 찾는 임베딩 검색 |
| ReRank | 1차로 넉넉히 찾은 후보를 **더 정밀한 모델로 다시 줄 세우기** |
| HyQE / HyDE | 가상 **질문** / 가상 **답변**을 LLM으로 만들어 검색에 활용 |

### 실행 시간 안내 (T4 GPU 기준)
- 임베딩: 여러 방식으로 인덱스를 만들기 때문에 합계 10~15분
- LLM 호출: HyQE·Atomic 인덱스 생성(각 수백 회, `max_concurrency=8` 병렬) + 채점 → 합계 20~30분
- 오래 걸리는 셀에는 ⏱ 표시를 했습니다

---
## 0. 실행 준비

### 0-1. 구글 드라이브 연결 + 라이브러리 설치
- 임베딩·리랭커 모델을 드라이브에 저장(캐시)해 두면 다음 실행부터 다시 받지 않음
- `GPU: Tesla T4`가 나오면 정상. `없음`이 나오면 **런타임 → 런타임 유형 변경 → T4 GPU** 후 다시 실행

In [6]:
# from google.colab import drive
# drive.mount('/content/drive')                                   # 구글 드라이브 연결

# import os, torch

# os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'      # 모델을 드라이브에 캐시
# print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 → 런타임 유형을 T4 GPU로 변경")

!pip install -q langchain langchain-community langchain-huggingface langchain-upstage langchain-milvus "pymilvus[milvus_lite]" pypdf sentence-transformers pandas rank_bm25 scikit-learn


[notice] A new release of pip is available: 23.0.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


### 0-2. 규정 데이터 내려받기 + API Key 설정
- 3장 실습과 같은 데이터: 명지대학교 규정 PDF 239개 (GitHub에서 자동 다운로드)
- Upstage API Key는 왼쪽 🔑 보안 비밀에 `UPSTAGE_API_KEY`로 등록하거나, 실행 시 직접 입력

In [10]:
from getpass import getpass
# from google.colab import userdata
import os
import zipfile, urllib.request

DATA_URL = 'https://github.com/bjkim2004/Vector_Graph/raw/main/%EB%AA%85%EC%A7%80%EB%8C%80%EA%B7%9C%EC%A0%95.zip'
ZIP_PATH = '명지대규정.zip'
DATA_DIR = '명지대규정'                     # 규정 PDF 폴더
CACHE_PATH = 'raw_docs.json'              # PDF 읽은 결과 저장 파일
DB_URI = 'mju_advanced.db'                   # Milvus Lite 벡터 DB 파일

if not os.path.isdir(DATA_DIR):                    # 없을 때만 내려받기
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)
print("규정 파일 수:", sum(len(files) for _, _, files in os.walk(DATA_DIR)))

os.environ['UPSTAGE_API_KEY'] = getpass('UPSTAGE_API_KEY: ')

규정 파일 수: 240
UPSTAGE_API_KEY: ········


### 0-3. 모델 준비
- `embeddings`: 글 → 숫자 벡터 (BGE-m3-ko, 1024차원)
- `llm`: 답변·질문 변환 등 **일하는 모델** (solar-mini4) / `judge`: **채점하는 모델** (solar-pro4, 자기 선호 편향을 줄이려고 분리)

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_upstage import ChatUpstage

embeddings = HuggingFaceEmbeddings(
    model_name="dragonkue/BGE-m3-ko",
    encode_kwargs={"normalize_embeddings": True, "batch_size": 32})   # 길이 1로 정규화 → 내적 = 코사인 유사도

llm = ChatUpstage(model="solar-mini4", temperature=0)      # 일하는 모델
judge = ChatUpstage(model="solar-pro4", temperature=0)     # 채점 모델

/home/elicer/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## 1. 공통 준비 — 3장 실습과 같은 코드

> Advanced RAG도 결국 '같은 문제지로 전후 점수를 비교'하는 것이 핵심입니다.
> 아래 셀들은 **3장 실습(3. RAG_구축과_성능평가)과 같은 코드**입니다. 설명은 짧게, 차례대로 실행만 하면 됩니다.

### 1-1. PDF 읽기 → 정리(정제)
- 규정 1개 = 문서(Document) 1개, 메타데이터(편·장·규정번호·규정명) 붙이기
- 페이지 번호 제거, 공백 정리, `제N조(` 앞에 빈 줄 넣기(조 단위로 자르는 기준)

In [12]:
import re, glob, json, time, math
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document

def parse_meta(path):
    """파일 경로 → 메타데이터(편·장·규정번호·규정명)"""
    p = Path(path)
    parts = p.relative_to(DATA_DIR).parts
    m = re.match(r"^([\d\-]+)\.?\s*(.+)$", p.stem)
    reg_id, title = (m.group(1), m.group(2).strip()) if m else (p.stem, p.stem)
    return {"part": parts[0], "chapter": parts[1] if len(parts) > 2 else "",
            "reg_id": reg_id, "title": title, "source": p.name}

if os.path.exists(CACHE_PATH):                                   # 저장된 결과가 있으면 바로 읽기
    raw_docs = [Document(**d) for d in json.load(open(CACHE_PATH, encoding="utf-8"))]
else:                                                            # 없으면 PDF 239개 읽기 (수 분)
    raw_docs = []
    for path in tqdm(sorted(glob.glob(f"{DATA_DIR}/**/*.pdf", recursive=True))):
        text = "\n".join(p.page_content for p in PyPDFLoader(path).load())
        raw_docs.append(Document(page_content=text, metadata=parse_meta(path)))
    json.dump([{"page_content": d.page_content, "metadata": d.metadata} for d in raw_docs],
              open(CACHE_PATH, "w", encoding="utf-8"), ensure_ascii=False)

def clean(text):
    text = re.sub(r"-\s+\d{1,3}\s+-", " ", text)                            # 페이지 번호 '- 5 -'
    text = re.sub(r"\s+", " ", text)                                         # 공백 정리
    text = re.sub(r"\s*(제\s?\d+\s?조(?:의\s?\d+)?\s?\()", r"\n\n\1", text)  # 조 앞에 빈 줄
    return text.strip()

docs = [Document(page_content=clean(d.page_content), metadata=d.metadata) for d in raw_docs]
print("문서 수:", len(docs), "/ 전체 글자 수:", sum(len(d.page_content) for d in docs))

/tmp/ipykernel_99/402967745.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
100%|██████████| 239/239 [02:11<00:00,  1.82it/s]

문서 수: 239 / 전체 글자 수: 1638899


### 1-2. 규정 맞춤 청킹 함수 `make_chunks`
- 개정 이력 주석 제거 → 조 경계 우선 분할 → 조 머리말 반복 → `[규정명]` 붙이기 → `chunk_id` 부여

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

NOTE = re.compile(                                               # 개정 이력 주석: (변경 2005.3.1) 등
    r"\s?[(\[]"
    r"[^()\[\]]*(?:변경|신설|삭제|개정|제정|시행|이동)"
    r"[^()\[\]]*(?:\d{2,4}\s?\.\s?\d{1,2}\s?\.\s?\d{1,2}|\d{2,4}\s*\.\s*|\d+)"
    r"[^()\[\]]*[)\]]")
ARTICLE_HEAD = re.compile(r"제\s?\d+\s?조(?:의\s?\d+)?\s?\([^)]*\)")   # 조 머리말: 제50조(학사과정의 졸업)

def strip_notes(text):
    return NOTE.sub("", text)

def make_chunks(docs, size=500, overlap=50, prefix=None):
    splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", ". ", " ", ""], chunk_size=size, chunk_overlap=overlap,
        keep_separator="end", add_start_index=True)
    chunks = []
    for d in docs:
        text = strip_notes(d.page_content)                                  # ① 주석 제거
        heads = [(m.start(), m.group()) for m in ARTICLE_HEAD.finditer(text)]
        for c in splitter.split_documents([Document(page_content=text, metadata=d.metadata)]):   # ② 조 경계 우선
            start = c.metadata.pop("start_index")
            head = next((h for p, h in reversed(heads) if p <= start), "")   # 청크가 속한 조
            body = c.page_content.strip()
            if head and not body.startswith(head.split("(")[0]):
                body = f"{head} … {body}"                                   # ③ 조 머리말 반복
            c.page_content = f"[{c.metadata['title']}] {body}"              # ④ 규정명 붙이기
            c.metadata["article"] = head.split("(")[0].replace(" ", "")
            chunks.append(c)
    for i, c in enumerate(chunks):
        c.metadata["chunk_id"] = f"{prefix or size}-{i}"
    return chunks

### 1-3. 채점용 문제지(Golden Set) — 26문항
- 3장의 25문항 + **다단계 계산 질문 q23** 1개 추가
  - q23: 졸업학점(134학점, 학칙 제50조) ÷ 학기당 최대 학점(18학점, 학칙 제31조) → 최소 8학기
  - 근거가 두 조항에 나뉘어 있어 **한 번 검색으로는 풀기 어려운 질문** → 고급 기법의 효과를 확인하는 용도

In [14]:
golden = [
    # ── 단일 문서 사실형 ──
    {"id": "q01", "type": "사실형", "question": "학사과정의 수업연한은 몇 년인가요?",
     "ground_truth": "4년이다. 다만 건축대학은 5년이다.", "evidence": [("2-1-1", "학사과정의 수업연한은 4년으로 한다")]},
    {"id": "q02", "type": "사실형", "question": "인공지능·소프트웨어융합대학 학생의 졸업 이수학점은?",
     "ground_truth": "134학점 이상 취득해야 졸업이 인정된다.", "evidence": [("2-1-1", "소프트웨어융합대학은 134학점")]},
    {"id": "q03", "type": "사실형", "question": "조기졸업을 하려면 평균평점이 얼마 이상이어야 하나요?",
     "ground_truth": "전체 성적 평균평점 4.0 이상(학·석사 연계과정은 3.5 이상)이고 졸업학점을 취득하면 한 학기 조기졸업할 수 있다.",
     "evidence": [("2-1-1", "한 학기 조기졸업을 할 수 있다")]},
    {"id": "q04", "type": "사실형", "question": "학사경고를 받는 성적 기준은 무엇인가요?",
     "ground_truth": "당해학기 평균평점 1.50 미만이면 학사경고를 받는다. 수강신청을 하지 않은 재학생도 학사경고를 받는다.",
     "evidence": [("2-1-1", "평균평점 1.50미만인 자에게는 학사경고")]},
    {"id": "q05", "type": "사실형", "question": "경영대학 학생은 한 학기에 최대 몇 학점까지 취득할 수 있나요?",
     "ground_truth": "17학점이다(계절수업, 채플 등은 산입하지 않음).", "evidence": [("2-1-1", "아너칼리지(전공자유대학) : 17학점")]},
    {"id": "q06", "type": "사실형", "question": "계절수업으로 취득할 수 있는 학점 한도는?",
     "ground_truth": "7학점 이내이다.", "evidence": [("2-1-1", "계절수업 취득학점은 7학점 이내로 한다")]},
    {"id": "q07", "type": "사실형", "question": "원격수업은 개설 교과목 학점의 몇 퍼센트까지 운영할 수 있나요?",
     "ground_truth": "총 개설 교과목 학점수의 100분의 20(20%)을 초과할 수 없다.", "evidence": [("2-1-1", "100분의 20을 초과할 수 없다")]},
    {"id": "q08", "type": "사실형", "question": "학부생은 재학 중 일반휴학을 통산 몇 학기까지 할 수 있나요?",
     "ground_truth": "6개 학기를 초과할 수 없다(의무군복무·육아·창업 휴학은 제외).", "evidence": [("2-2-1", "일반휴학은 6개 학기를 초과할 수 없다")]},
    {"id": "q09", "type": "사실형", "question": "재수강은 어떤 성적의 과목에 가능하고 최고 몇 점까지 받을 수 있나요?",
     "ground_truth": "C+ 이하 과목만 학기당 최대 2과목, 과목당 최대 3회까지 가능하며 최고 성적은 A0이다.",
     "evidence": [("2-2-1", "재수강시 취득가능 최고 성적은 A0")]},
    {"id": "q10", "type": "사실형", "question": "상대평가 시 A학점 비율은 어떻게 되나요?",
     "ground_truth": "A학점 10~40%, B학점 20~40%, C~F학점 20~70%를 원칙으로 하며 A·B학점을 80%까지 상향할 수 있다.",
     "evidence": [("2-2-1", "A학점 10-40%")]},
    {"id": "q11", "type": "사실형", "question": "학생 징계의 종류에는 무엇이 있나요?",
     "ground_truth": "경고, 근신, 유기정학, 무기정학, 제적으로 구분한다.", "evidence": [("3-2-3", "경고, 근신, 유기정학, 무기정학, 제적으로 구분한다")]},
    {"id": "q12", "type": "사실형", "question": "학부생은 도서관에서 책을 몇 권, 며칠 동안 빌릴 수 있나요?",
     "ground_truth": "20책, 14일이다.", "evidence": [("5-1-1", "학부과정 학생 : 20책 14일")]},
    {"id": "q13", "type": "사실형", "question": "도서 반납이 연체되면 어떤 제재를 받나요?",
     "ground_truth": "반납 시까지 대출이 중지되고, 연체일수의 2배 기간 동안 대출이 중지된다. 180일을 초과하면 분실로 간주해 변상한다.",
     "evidence": [("5-1-1", "연체일수의 2배일에 해당하는 기간동안 대출을 중지")]},
    {"id": "q14", "type": "사실형", "question": "명예졸업은 어떤 사람에게 수여하나요?",
     "ground_truth": "재학 당시 품행이 방정했으나 특별한 사정이나 불의의 사고로 학업을 마치지 못한 사람 중 사회·국가 또는 대학 발전에 공헌했거나 명예졸업심사위원회 심의를 통과한 사람이다.",
     "evidence": [("3-1-36", "불의의 사고로 인하여 학업을 마치지 못한 사람")]},
    {"id": "q15", "type": "사실형", "question": "학생단체를 등록하려면 정규회원이 몇 명 이상 있어야 하나요?",
     "ground_truth": "20명 이상의 정규회원이 필요하다.", "evidence": [("3-2-2", "20명 이상의 입회원서")]},
    {"id": "q16", "type": "사실형", "question": "학생이 교내 집회를 열려면 언제까지 허가를 신청해야 하나요?",
     "ground_truth": "집회 및 행사 7일 전까지 학생처장에게 허가 신청서를 제출해 허가를 받아야 한다.",
     "evidence": [("3-2-1", "집회 및 행사예정 7일전까지")]},
    {"id": "q17", "type": "사실형", "question": "교수 연구년은 신규 임용 후 몇 년을 근무해야 신청할 수 있나요?",
     "ground_truth": "신규 임용 후 6년 이상 근무하면 6월 또는 1년의 연구년을 신청할 수 있다.",
     "evidence": [("4-2-11", "신규 임용 후 6년 이상 근무한 교원")]},
    {"id": "q18", "type": "사실형", "question": "연구년을 다녀온 교원은 복귀 후 얼마 동안 근무해야 하나요?",
     "ground_truth": "연구년 기간의 3배수에 해당하는 기간 동안 근무해야 한다.", "evidence": [("4-2-11", "연구년 기간의 3배수")]},
    {"id": "q19", "type": "사실형", "question": "교내에서 전동킥보드 음주운전으로 보는 혈중알코올농도 기준은?",
     "ground_truth": "혈중알코올농도 0.03% 이상이다.", "evidence": [("4-4-32", "혈중알코올농도가 0.03퍼센트 이상")]},
    {"id": "q20", "type": "사실형", "question": "개인 소유 전동킥보드를 교내 등록하면 유효기간은 얼마인가요?",
     "ground_truth": "등록일로부터 1년이며, 종료일이 학기 중이면 해당 학기 말까지이다.", "evidence": [("4-4-32", "운행 유효기간은 등록일로부터 1년")]},
    # ── 복수 근거 종합형 ──
    {"id": "q21", "type": "종합형", "question": "학부생은 한 번에 몇 학기까지 휴학할 수 있고, 재학 중 일반휴학은 통산 몇 학기까지 가능한가요?",
     "ground_truth": "1회에 두 학기를 초과할 수 없고, 재학 중 일반휴학은 통산 6개 학기를 초과할 수 없다.",
     "evidence": [("2-1-1", "휴학은 1회에 두 학기를 초과하지 못하며"), ("2-2-1", "일반휴학은 6개 학기를 초과할 수 없다")]},
    {"id": "q22", "type": "종합형", "question": "성적을 인정받으려면 수업에 얼마나 출석해야 하고, 군복무 휴학생은 기준이 어떻게 다른가요?",
     "ground_truth": "수업일수의 5분의 4 이상 출석해야 성적이 인정된다. 의무군복무·육아·창업 휴학의 경우 3분의 2 기준이 적용된다.",
     "evidence": [("2-1-1", "수업일수 5분의 4 이상을 출석한 자에 한하여"), ("2-2-1", "3분의 2미만일 경우 성적을 인정하지 아니한다")]},
    {"id": "q23", "type": "종합형", "question": "인공지능·소프트웨어융합대학 학생이 매 학기 최대 학점만 수강하면 졸업까지 최소 몇 학기가 필요한가요?",
     "ground_truth": "졸업학점 134학점 ÷ 학기당 최대 18학점 = 7.4이므로 최소 8개 학기가 필요하다(계절수업 제외).",
     "evidence": [("2-1-1", "소프트웨어융합대학은 134학점"), ("2-1-1", "소프트웨어융합대학 : 18학점")]},
    # ── 정답 없는 질문 ──
    {"id": "n01", "type": "정답없음", "question": "2027학년도 명지대학교 등록금 인상률은 몇 퍼센트인가요?",
     "ground_truth": "규정에 해당 정보가 없으므로 알 수 없다고 답해야 한다.", "evidence": []},
    {"id": "n02", "type": "정답없음", "question": "이번 주 학생식당 점심 메뉴는 무엇인가요?",
     "ground_truth": "규정에 해당 정보가 없으므로 알 수 없다고 답해야 한다.", "evidence": []},
    {"id": "n03", "type": "정답없음", "question": "기숙사 입사 신청 시 통금 시간은 몇 시인가요?",
     "ground_truth": "규정에 해당 정보가 없으므로 알 수 없다고 답해야 한다.", "evidence": []},
]
GOLD_REGS = sorted({src for g in golden for src, _ in g["evidence"]})     # 정답 근거가 있는 규정 9개
pd.DataFrame(golden)["type"].value_counts()

type
사실형     20
종합형      3
정답없음     3
Name: count, dtype: int64

### 1-4. 채점 도구 — 검색 지표 6개 + 응답 지표 3개
- 검색 지표 함수는 3장에서 직접 만든 것과 같음 (LangChain에는 내장 함수가 없음)
- `label_docs`: 검색된 청크마다 '근거N' 또는 '무관' 이름표 → 지표 함수에 그대로 입력
- `eval_retrieval(search_fn, k)`: **검색 방법만 바꿔 끼워서** 문제지 전체의 검색 점수표를 만듦

In [15]:
def norm(text):
    return re.sub(r"\s+", "", text)                    # 공백 모두 제거 (비교용)

# ── 검색 지표 6개 (입력: retrieved=검색 결과 목록, relevant=정답 모음, k=상위 몇 개) ──
def hit_at_k(retrieved, relevant, k):
    return int(any(chunk in relevant for chunk in retrieved[:k]))

def precision_at_k(retrieved, relevant, k):
    return sum(1 for chunk in retrieved[:k] if chunk in relevant) / k

def recall_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & set(relevant)) / len(relevant) if relevant else 0.0

def reciprocal_rank(retrieved, relevant, k):
    for rank, chunk in enumerate(retrieved[:k], start=1):
        if chunk in relevant:
            return 1 / rank
    return 0.0

def ndcg_at_k(retrieved, relevant, k):
    dcg, seen = 0.0, set()
    for rank, chunk in enumerate(retrieved[:k], start=1):
        if chunk in relevant and chunk not in seen:           # 같은 근거는 처음 한 번만
            dcg += 1 / math.log2(rank + 1)
            seen.add(chunk)
    idcg = sum(1 / math.log2(rank + 1) for rank in range(1, min(len(relevant), k) + 1))
    return dcg / idcg if idcg else 0.0

def context_precision_at_k(retrieved, relevant, k):
    hits, total = 0, 0.0
    for rank, chunk in enumerate(retrieved[:k], start=1):
        if chunk in relevant:
            hits += 1
            total += hits / rank
    return total / hits if hits else 0.0

def retrieval_metrics(retrieved, relevant, k):
    return {"hit": hit_at_k(retrieved, relevant, k), "precision": precision_at_k(retrieved, relevant, k),
            "recall": recall_at_k(retrieved, relevant, k), "mrr": reciprocal_rank(retrieved, relevant, k),
            "ndcg": ndcg_at_k(retrieved, relevant, k), "ctx_precision": context_precision_at_k(retrieved, relevant, k)}

METRICS = ["hit", "precision", "recall", "mrr", "ndcg", "ctx_precision"]

# ── 검색된 청크에 이름표 붙이기 ──
def label_docs(docs, item):
    labels = []
    for d in docs:
        label = "무관"
        for n, (src, ev) in enumerate(item["evidence"], start=1):
            if d.metadata["reg_id"] == src and norm(ev) in norm(d.page_content):
                label = f"근거{n}"
                break
        labels.append(label)
    return labels

def relevant_set(item):
    return {f"근거{n}" for n in range(1, len(item["evidence"]) + 1)}

def format_docs(docs):
    return "\n\n".join(f"<{i+1}> {d.page_content}" for i, d in enumerate(docs))

def eval_retrieval(search_fn, k=4, items=None):
    """search_fn(질문, k) → 청크 목록. 문제지 전체의 검색 점수표(DataFrame)"""
    rows = []
    for item in (items or golden):
        if not item["evidence"]:                       # 정답 없는 질문은 제외
            continue
        t0 = time.time()
        found = search_fn(item["question"], k)
        ms = (time.time() - t0) * 1000
        rows.append({"id": item["id"], "type": item["type"],
                     **retrieval_metrics(label_docs(found, item), relevant_set(item), k),
                     "ctx_chars": len(format_docs(found)), "search_ms": ms})
    return pd.DataFrame(rows)

def compare(results):
    """{이름: eval_retrieval 결과} → 평균 비교표"""
    return pd.DataFrame({name: df[METRICS + ["ctx_chars", "search_ms"]].mean() for name, df in results.items()}).T.round(3)

### 1-5. 답변 만들기와 응답 채점
- `RAG_PROMPT`: 컨텍스트 안의 내용만 사용 / 근거 없으면 '찾을 수 없습니다' / 출처 표기
- `make_rag(search_fn, k)`: 검색 방법을 받아 **질문 → 검색 → 답변** 함수를 만들어 줌
- `evaluate(pipeline)`: 질문마다 답변을 만들고 검색 지표 + 응답 지표(LLM 채점)를 계산

In [16]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "너는 명지대학교 규정 안내 도우미다. 반드시 [컨텍스트]에 있는 내용만 근거로 한국어로 간결하게 답하라.\n"
               "컨텍스트에 근거가 없으면 추측하지 말고 '규정에서 찾을 수 없습니다'라고 답하라.\n"
               "계산이 필요하면 근거 숫자와 계산 과정을 보여라.\n"
               "답변 마지막 줄에 (출처: 규정명 제N조) 형식으로 근거를 적어라."),
    ("human", "[컨텍스트]\n{context}\n\n[질문]\n{question}"),
])
answer_chain = RAG_PROMPT | llm | StrOutputParser()

def make_rag(search_fn, k=4):
    """검색 방법 search_fn을 받아 '질문 → 검색 → 답변' 함수를 만들어 돌려줌"""
    def pipeline(question):
        t0 = time.time()
        docs = search_fn(question, k)
        answer = answer_chain.invoke({"context": format_docs(docs), "question": question})
        return {"answer": answer, "docs": docs, "total_ms": (time.time() - t0) * 1000, "llm_calls": 1}
    return pipeline

# ── 응답 채점 (채점 모델 judge 사용) ──
class Claims(BaseModel):
    claims: list[str] = Field(description="답변을 구성하는 개별 사실 주장 목록 (출처 표기 문장은 제외)")
    supported: list[bool] = Field(description="각 주장이 컨텍스트로 뒷받침되면 true")

class GenQuestions(BaseModel):
    questions: list[str] = Field(description="답변을 보고 역으로 추정한 질문 3개")

class Verdict(BaseModel):
    correct: bool = Field(description="응답이 정답과 사실적으로 일치하면 true")
    reason: str = Field(description="판정 이유 한 문장")

def faithfulness(answer, docs):
    r = judge.with_structured_output(Claims, method="function_calling").invoke(
        f"[컨텍스트]\n{format_docs(docs)}\n\n[답변]\n{answer}\n\n"
        "답변을 개별 사실 주장으로 나누고, 각 주장이 컨텍스트로 뒷받침되는지 판정하라. "
        "컨텍스트의 숫자로 계산한 결과는 뒷받침되는 것으로 본다. "
        "'찾을 수 없다'처럼 정보가 없다는 답변은 주장이 없는 것으로 본다.")
    return float(np.mean(r.supported)) if r.supported else 1.0

def answer_relevancy(question, answer):
    r = judge.with_structured_output(GenQuestions, method="function_calling").invoke(
        f"[답변]\n{answer}\n\n이 답변이 대답하고 있는 질문을 한국어로 3개 만들어라.")
    qv = np.array(embeddings.embed_query(question))
    gv = np.array(embeddings.embed_documents(r.questions or [answer]))
    return float((gv @ qv).mean())

def correctness(question, answer, ground_truth):
    r = judge.with_structured_output(Verdict, method="function_calling").invoke(
        f"[질문]\n{question}\n\n[정답]\n{ground_truth}\n\n[응답]\n{answer}\n\n"
        "응답이 정답의 핵심 사실(숫자, 기간, 대상 등)과 일치하는지 판정하라. 표현 차이나 추가 설명은 허용한다. "
        "정답이 '알 수 없다'인 경우 응답도 정보가 없다고 답해야 true이다.")
    return r.correct, r.reason

def evaluate(pipeline, items=None, k=4, relevancy=True):
    """pipeline(질문) → {answer, docs, total_ms, llm_calls}. 문항별 검색·응답 점수표"""
    rows = []
    for item in tqdm(items or golden):
        out = pipeline(item["question"])
        row = {"id": item["id"], "type": item["type"], "answer": out["answer"]}
        if item["evidence"]:
            row.update(retrieval_metrics(label_docs(out["docs"], item), relevant_set(item), max(k, len(out["docs"]))))
        row["faithfulness"] = faithfulness(out["answer"], out["docs"])
        if relevancy:
            row["relevancy"] = answer_relevancy(item["question"], out["answer"])
        row["correct"], row["reason"] = correctness(item["question"], out["answer"], item["ground_truth"])
        row.update(ctx_chars=len(format_docs(out["docs"])), total_ms=out["total_ms"], llm_calls=out.get("llm_calls", 1))
        rows.append(row)
    return pd.DataFrame(rows)

def summary(df):
    cols = [c for c in METRICS + ["faithfulness", "relevancy", "correct", "ctx_chars", "total_ms", "llm_calls"] if c in df]
    s = df[cols].mean(numeric_only=True)
    s["negative_rejection"] = df.loc[df["type"] == "정답없음", "correct"].mean()
    s["multi_correct"] = df.loc[df["type"] == "종합형", "correct"].mean()     # 종합형(다중 근거) 정답률
    return s

### 1-6. 벡터 DB 저장 함수 + 기본 검색
- `build_store`: 청크를 임베딩해 Milvus 컬렉션에 저장 (3장과 같음)
- `memory_store`: 작은 실험용 **메모리 저장소** (Milvus와 같은 코사인 유사도 → 검색 순위 동일)
- `dense_fn(store)`: 저장소를 받아 `search(질문, k)` 검색 함수를 만들어 줌 → `eval_retrieval`에 그대로 넣을 수 있음

In [17]:
import logging
from langchain_milvus import Milvus
from langchain_core.vectorstores import InMemoryVectorStore

logging.getLogger("grpc._server").setLevel(logging.CRITICAL)      # Milvus Lite 불필요 로그 숨김

def build_store(chunks, name):
    """청크 → 임베딩 → Milvus 컬렉션 저장"""
    return Milvus.from_documents(
        chunks, embeddings, collection_name=name,
        connection_args={"uri": DB_URI},
        index_params={"index_type": "FLAT", "metric_type": "COSINE"},
        ids=[c.metadata["chunk_id"] for c in chunks], drop_old=True)

def memory_store(docs):
    """실험용 메모리 벡터 저장소 (임베딩 진행률 표시)"""
    embeddings.show_progress = True
    vs = InMemoryVectorStore(embeddings)
    vs.add_documents(docs, ids=[d.metadata["chunk_id"] for d in docs])
    embeddings.show_progress = False
    return vs

def dense_fn(store):
    """저장소 → search(질문, k) 함수"""
    def search(q, k):
        return store.similarity_search(q, k=k)
    return search

---
# Part A. 검색 전 절차 ① — 원천 데이터의 정비

## A-1. 불필요한 데이터 제거 — '부칙' 걸러내기
- 강의자료: *"필요한 Data의 대상은 의외로 적을 수 있다"* → 제거 대상 예시: 오래된 이력, 목차·머리글, 주석 처리 내용
- 규정 PDF의 **부칙(附則)** = 개정할 때마다 붙는 '언제부터 시행한다'는 기록 → 질문과 거의 상관없는 **오래된 이력(OLD 데이터)**
  - 예: 명지대학교 학칙 9만 5천 자 중 **약 7만 자가 부칙** ("본 개정 학칙은 1976년 4월 15일부터 시행한다" …)
- 단, 부칙 뒤의 **[별표]** (입학정원표, 학점표 등)는 실제 정보라서 남김

In [18]:
BUCHIK = re.compile(r"부\s?칙")                      # '부칙' 또는 '부 칙'
BYEOLPYO = re.compile(r"[\[<【(]\s?별\s?표")          # [별표 …] 시작 위치

def strip_buchik(text):
    """첫 번째 '부칙'부터 [별표] 직전까지를 지움 (별표가 없으면 끝까지)"""
    m = BUCHIK.search(text)
    if not m:
        return text
    rest = text[m.start():]
    b = BYEOLPYO.search(rest)
    return text[:m.start()] + (" " + rest[b.start():] if b else "")

docs_clean = [Document(page_content=strip_buchik(d.page_content), metadata=d.metadata) for d in docs]

before, after = sum(len(d.page_content) for d in docs), sum(len(d.page_content) for d in docs_clean)
print(f"전체 글자 수: {before:,} → {after:,} ({after / before:.0%})")
hakchik = next(d for d in docs if d.metadata["reg_id"] == "2-1-1")
cut = BUCHIK.search(hakchik.page_content).start()
print(f"학칙: 전체 {len(hakchik.page_content):,}자 중 부칙 시작 위치 {cut:,}자")
print("지워지는 부분 예시:", hakchik.page_content[cut:cut + 150], "...")

전체 글자 수: 1,638,899 → 1,457,945 (89%)
학칙: 전체 98,496자 중 부칙 시작 위치 26,721자
지워지는 부분 예시: 부 칙 본 학칙 실시에 필요한 세칙은 총장이 이를 따로 정한다. 부 칙 본 학칙은 1970년 3월 1일부터 시행한다. 부 칙 본 학칙은 1975년 8월 8일부터 시행한다. 부 칙 본 개정 학칙은 1976년 4월 15일부터 시행한다. 부 칙 본 개정 학칙은 1979년 5 ...


### 정답 근거가 지워지지 않았는지 확인 → 청크 만들기
- 데이터를 지울 때는 **꼭 필요한 정보까지 지우지 않았는지** 먼저 확인 (모두 `O`면 안전)

In [19]:
clean_by_id = {d.metadata["reg_id"]: norm(d.page_content) for d in docs_clean}
missing = [(g["id"], ev) for g in golden for src, ev in g["evidence"] if norm(ev) not in clean_by_id.get(src, "")]
print("부칙 제거 후 사라진 정답 근거:", missing if missing else "없음 → O")

chunks_base = make_chunks(docs, prefix="base")          # 3장과 같은 청크
chunks_clean = make_chunks(docs_clean, prefix="clean")  # 부칙 제거 후 청크
print("청크 수:", len(chunks_base), "→", len(chunks_clean))

부칙 제거 후 사라진 정답 근거: 없음 → O
청크 수: 4150 → 3708


### 전체 규정으로 비교: 부칙 제거 전 vs 후 ⏱ (임베딩 약 6~8분)
- 두 컬렉션을 만들고 같은 문제지로 검색 점수 비교
- 기대: 노이즈 청크가 줄어 **MRR·NDCG(정답이 위로 올라오는 정도)** 개선

In [20]:
store_base = build_store(chunks_base, "mju_base")
store_clean = build_store(chunks_clean, "mju_clean")     # 이후 실습의 기본 저장소
dense_search = dense_fn(store_clean)

results_A = {"3장 기본": eval_retrieval(dense_fn(store_base), 4),
             "부칙 제거": eval_retrieval(dense_search, 4)}
compare(results_A)

,hit,precision,recall,mrr,ndcg,ctx_precision,ctx_chars,search_ms
3장 기본,0.87,0.239,0.87,0.728,0.758,0.717,1717.609,390.690
부칙 제거,0.87,0.239,0.87,0.728,0.758,0.717,1696.304,333.049


## A-2. LLM을 통한 정련 — 긴 조문을 요약·태그로 정리
- 강의자료 3단계: ① 원시 데이터에서 유용한 정보 추출 → ② 요약·핵심 사실 분리로 **정보 밀도 향상** → ③ 메타 정보와 함께 저장
- `with_structured_output`: LLM 결과를 **정해진 모양(Pydantic 클래스)** 으로 받기
- 결과(요약·핵심 사실·대상·키워드)는 청크의 **메타데이터**로 저장해 검색·필터에 활용할 수 있음

In [21]:
from typing import Literal

class Refined(BaseModel):
    summary: str = Field(description="조문을 한두 문장으로 요약")
    key_facts: list[str] = Field(description="숫자·기간·대상이 들어간 핵심 사실 목록")
    target: Literal["학부생", "대학원생", "교원", "직원", "전체"] = Field(description="이 조문이 주로 적용되는 대상")
    keywords: list[str] = Field(description="검색에 도움이 되는 키워드 3~5개")

refiner = llm.with_structured_output(Refined, method="function_calling")

sample = next(c for c in chunks_clean if c.metadata["reg_id"] == "2-1-1" and c.metadata["article"] == "제31조")
print("[원문]", sample.page_content[:400], "\n")
r = refiner.invoke(f"다음 대학 규정 조문을 정련하라.\n\n{sample.page_content}")
print("[요약]", r.summary)
print("[핵심 사실]", r.key_facts)
print("[대상]", r.target, "/ [키워드]", r.keywords)

[원문] [명지대학교 학칙] 제31조(학사과정의 학기당 취득학점) ①학사과정 학생의 학기당 최대 취득학점은 다 음과 같다. 다만, 계절수업, 채플, 봉사학점, 교양실습학점, 교육봉사학점, 군사학 학 점은 이에 산입 하지 아니한다. 1.인문대학, 사회과학대학, 경영대학, 미디어·휴먼라이프대학, 미래융합대학, 화 학·생명과학대학, 스포츠·예술대학, 아너칼리지(전공자유대학) : 17학점 2.스마트시스템공과대학, 반도체·ICT대학, 건축대학, 인공지능·소프트웨어융합대 학 : 18학점 다만, 외부지원사업 등 을 통한 특수 프로그램 운영시 별도로 정할 수 있으며 세부사항은 학칙시행규칙 (학사과정)에 의한다. ②계절수업 취득학점은 7학점 이내로 한다. ③직전학기 평점평균이 4.0이상인 자는 학점을 초과하여 이수할 수 있으며 이에 

[요약] 명지대학교 학칙 제31조는 학사과정 학생의 학기당 최대 취득학점을 단과대학별로 17학점 또는 18학점으로 정하고, 계절수업·채플·봉사학점 등은 산입하지 않으며, 계절수업 취득학점은 7학점 이내, 직전학기 평점평균 4.0 이상인 자는 초과 이수가 가능하도록 규정한다.
[핵심 사실] ['명지대학교 학칙 제31조 제1항: 학사과정 학생의 학기당 최대 취득학점', '인문대학, 사회과학대학, 경영대학, 미디어·휴먼라이프대학, 미래융합대학, 화학·생명과학대학, 스포츠·예술대학, 아너칼리지(전공자유대학): 17학점', '스마트시스템공과대학, 반도체·ICT대학, 건축대학, 인공지능·소프트웨어융합대학: 18학점', '계절수업, 채플, 봉사학점, 교양실습학점, 교육봉사학점, 군사학 학점은 최대 취득학점에 산입하지 않음', '제2항: 계절수업 취득학점은 7학점 이내', '제3항: 직전학기 평점평균 4.0 이상인 자는 학점 초과 이수 가능', '제4항: 학점취득에 관한 세부사항은 학칙시행규칙(학사과정)에 위임']
[대상] 전체 / [키워드] ['명지대학교', '학칙', '제31조', '학사과정', '학기당 취득학점', '최대 취득학점', '계절수업', '평

## A-3. 중복 데이터 그룹화 — 비슷한 청크 묶기
- 같은 내용이 여러 규정에 반복되면(예: 학칙과 시행규칙의 '휴학') 검색 결과 자리를 **중복 정보가 차지**함
- 강의자료 해결 프로세스: ① **그룹화**(K-means 클러스터링) → ② **병합**(LLM으로 하나의 청크로) → ③ **정제**(중복 제거)
- 여기서는 정답 근거가 있는 규정 9개(`GOLD_REGS`)의 청크로 실습 (빠르게 확인하기 위해)

In [22]:
from sklearn.cluster import KMeans

sub_chunks = [c for c in chunks_clean if c.metadata["reg_id"] in GOLD_REGS]
vecs = np.array(embeddings.embed_documents([c.page_content for c in sub_chunks]))   # 청크 벡터
print("실습 청크 수:", len(sub_chunks))

# ① 매우 비슷한 청크 쌍 찾기 (코사인 유사도 0.93 이상, 서로 다른 규정)
sim = vecs @ vecs.T
pairs = [(sim[i, j], i, j) for i in range(len(sub_chunks)) for j in range(i + 1, len(sub_chunks))
         if sim[i, j] >= 0.93 and sub_chunks[i].metadata["reg_id"] != sub_chunks[j].metadata["reg_id"]]
pairs.sort(reverse=True)
print(f"서로 다른 규정 사이의 중복 의심 쌍: {len(pairs)}개")
for s, i, j in pairs[:3]:
    print(f"\n유사도 {s:.3f}")
    print("  A:", sub_chunks[i].page_content[:110])
    print("  B:", sub_chunks[j].page_content[:110])

실습 청크 수: 386
서로 다른 규정 사이의 중복 의심 쌍: 0개


In [23]:
# ② K-means로 그룹화 → 여러 규정이 섞인 그룹 찾기
kmeans = KMeans(n_clusters=40, random_state=0, n_init=10).fit(vecs)
groups = {}
for c, label in zip(sub_chunks, kmeans.labels_):
    groups.setdefault(label, []).append(c)
mixed = sorted(groups.values(), key=lambda g: -len({c.metadata["reg_id"] for c in g}))[0]
print("여러 규정이 섞인 그룹:", sorted({c.metadata["title"] for c in mixed}), f"/ 청크 {len(mixed)}개")

# ③ LLM으로 그룹 대표 청크 3개를 하나로 병합 (데모)
merge_prompt = ("다음은 서로 비슷한 대학 규정 청크들이다. 중복 내용은 한 번만 남기고, "
                "규정마다 다른 점(대상·숫자)은 구분해서 하나의 정리된 문단으로 병합하라.\n\n" + format_docs(mixed[:3]))
print("\n[병합 결과]\n", llm.invoke(merge_prompt).content)

여러 규정이 섞인 그룹: ['학생단체 등록과 운영에 관한 규정', '학생생활 규정', '학칙 시행규칙(학사과정)'] / 청크 19개

[병합 결과]
 아래는 주어진 규정 청크들을 중복 내용은 한 번만 남기고, 규정마다 다른 점(대상·숫자)은 구분해서 하나의 정리된 문단으로 병합한 결과입니다.

---

**학칙 시행규칙(학사과정) 관련 규정**

- **제111조(회원자격·권리의무)** 회원은 우리 대학교에 재학하는 학생으로 하며, 회원은 본 회의 자치활동에 참여할 권리와 회칙을 준수하고 소정의 회비를 납부하여야 한다.  
- **제112조(임무·기능)** 본 회의 임무 및 기능은 다음 각 호와 같다.  
  1. 문화창달을 위한 각종 학술, 예술 및 체육활동  
  2. 의료 및 기술봉사 등 각 분야의 봉사활동  
  3. 기타 학생 자치활동  
- **제113조(기구·조직)** 본 회는 단과대학학생회와 필요한 부서를 둘 수 있다.  
- **제114조(선임방법·임기)** 본 회의 대표는 직접선거에 의하여 선출하며, 그 임기는 당해 학년도 1년으로 한다. 다만, 학생회장이 휴학, 퇴학 및 탄핵 등의 사유로 궐위된 때에는 부학생회장이 그 직을 대행하고, 학생회장과 부학생회장이 모두 궐위된 경우 잔여임기가 6개월 이상일 때에는 궐위된 날로부터 1개월 이내에 보궐선거를 실시하되, 잔여임기가 6개월 미만인 경우에는 본 회의 임원 중에서 호선한다.  
- **제115조(피선거권)** ① 본 회의 피선거권은 다음 각 호와 같다.  
  1. 학생회장 및 부회장은 선거공고일 현재 6학기 이수 중인 재학생으로 하고, 직전학기 성적 평점이 C+(2.5) 이상인 재학생  
  2. 유기정학 이상의 징계 또는 집행유예 이상의 형사처벌을 받지 않은 자  
  3. 회원의 1/10 이상의 추천을 받은 자  
  ② 본 회의 기타 부서 대표에 대하여는 전항을 준용한다.  
- **제116조(활동·간행물)** 본 회는 다음 각 호의 사항에 대하여 총장의 사전 승인을 받아야 한다.  


> **주의**: 병합한 청크로 원본을 바로 바꾸면 출처(어느 규정 몇 조) 추적이 어려워짐 → 실무에서는 **원본은 남기고** 병합본을 별도로 저장하거나, 검색 결과에서 중복을 걸러내는 방식을 함께 씀

---
# Part B. 검색 전 절차 ② — Retrieval 전략 (인덱스를 어떻게 만들까?)

- 강의자료 핵심: **청킹과 검색은 모든 RAG의 필수 단계, 청킹 선택이 곧 검색 품질을 결정**
- 공정한 비교를 위해 **실험용 지식베이스**를 고정: 정답 근거가 있는 규정 9개(부칙 제거 후)
- 같은 문제지·같은 임베딩 모델로 **인덱스 방식만 바꿔서** 검색 점수 비교

| 전략 | 검색 대상 | LLM에 넘기는 것 |
|---|---|---|
| 기본 청킹 | 500자 청크 | 같은 청크 |
| 계층 인덱스 | 규정(큰 단위) → 그 안의 청크 | 청크 |
| Parent-Child (Small to Big) | 작은 자식 청크(200자) | 큰 부모 청크(1000자) |
| Sentence Window | 문장 하나 | 문장 + 앞뒤 2문장 |
| HyQE (가상 질문) | LLM이 미리 만든 질문 | 질문이 가리키는 청크 |
| HyDE (가상 문서) | LLM이 만든 가상 답변으로 검색 | 실제 청크 |
| Atomic 청킹 | 원자 단위 사실의 합성 질문 | 연관된 청크 |

In [24]:
sub_docs = [d for d in docs_clean if d.metadata["reg_id"] in GOLD_REGS]
sub_store = memory_store(sub_chunks)                     # 기본 청킹 (실험용)
chunk_by_id = {c.metadata["chunk_id"]: c for c in sub_chunks}

results_B = {"기본 청킹": eval_retrieval(dense_fn(sub_store), 4)}
compare(results_B)

Batches: 100%|██████████| 13/13 [00:04<00:00,  2.91it/s]


,hit,precision,recall,mrr,ndcg,ctx_precision,ctx_chars,search_ms
기본 청킹,1.0,0.272,0.978,0.833,0.862,0.826,1658.957,46.749


## B-1. 계층 인덱스 검색 (Hierarchical Index)
- 1단계: **규정 단위**(큰 단위)로 먼저 찾기 → 규정마다 '규정명 + 조 제목 목록'을 하나의 요약 문서로 만들어 저장
- 2단계: 고른 규정 2개 **안에서만** 청크 검색 → 엉뚱한 규정의 청크가 끼어들지 못함

In [25]:
reg_summaries = []
for d in sub_docs:
    heads = list(dict.fromkeys(h.split(")")[0] + ")" for h in ARTICLE_HEAD.findall(d.page_content)))   # 조 제목(중복 제거)
    reg_summaries.append(Document(page_content=f"[{d.metadata['title']}] " + " ".join(heads)[:3000],
                                  metadata={**d.metadata, "chunk_id": "reg-" + d.metadata["reg_id"]}))
reg_store = memory_store(reg_summaries)

def hier_search(q, k):
    regs = [d.metadata["reg_id"] for d in reg_store.similarity_search(q, k=2)]     # 1단계: 규정 2개 고르기
    return sub_store.similarity_search(q, k=k, filter=lambda doc: doc.metadata["reg_id"] in regs)   # 2단계

print("질문: 학부생 도서 대출 권수 →", [d.metadata["title"] for d in reg_store.similarity_search("학부생 도서 대출 권수", k=2)])
results_B["계층 인덱스"] = eval_retrieval(hier_search, 4)

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.39it/s]


질문: 학부생 도서 대출 권수 → ['도서관 규정', '학생생활 규정']


## B-2. Small to Big ① Parent-Child 검색
- **작은 자식 청크(200자)로 검색** → 짧아서 질문과 정밀하게 매칭
- 찾은 자식의 **부모 청크(1000자)를 LLM에 전달** → 앞뒤 문맥이 풍부
- 주의: LLM에 넘기는 글자 수(토큰)가 늘어남 → `ctx_chars`도 함께 확인

In [26]:
parents = make_chunks(sub_docs, size=1000, overlap=100, prefix="parent")      # 큰 부모 청크
parent_by_id = {p.metadata["chunk_id"]: p for p in parents}

child_splitter = RecursiveCharacterTextSplitter(separators=[". ", " ", ""], chunk_size=200, chunk_overlap=20)
children = []
for p in parents:
    for j, piece in enumerate(child_splitter.split_text(p.page_content)):
        children.append(Document(page_content=f"[{p.metadata['title']}] {piece}",
                                 metadata={**p.metadata, "parent_id": p.metadata["chunk_id"],
                                           "chunk_id": f"{p.metadata['chunk_id']}-c{j}"}))
print(f"부모 {len(parents)}개 / 자식 {len(children)}개")
child_store = memory_store(children)

def parent_child_search(q, k):
    out, seen = [], set()
    for child in child_store.similarity_search(q, k=k * 5):       # 자식을 넉넉히 찾고
        pid = child.metadata["parent_id"]
        if pid not in seen:                                         # 부모로 바꿔서 중복 없이 k개
            seen.add(pid)
            out.append(parent_by_id[pid])
        if len(out) == k:
            break
    return out

results_B["Parent-Child"] = eval_retrieval(parent_child_search, 4)

부모 189개 / 자식 1044개


Batches: 100%|██████████| 33/33 [00:05<00:00,  6.21it/s]


## B-3. Small to Big ② Sentence Window 검색
- **문장 하나 단위**로 검색 (가장 정밀) → 찾은 문장의 **앞뒤 2문장(창, window)** 을 붙여서 전달
- 문장 나누기: `…다.` 뒤, 항 번호(①②③) 앞에서 자름

In [27]:
def split_sentences(text):
    parts = re.split(r"(?<=다\.)\s+|(?=[①-⑳])|\n\n", text)
    return [p.strip() for p in parts if len(p.strip()) > 5]

sent_by_reg, sentences = {}, []
for d in sub_docs:
    sents = split_sentences(strip_notes(d.page_content))
    sent_by_reg[d.metadata["reg_id"]] = sents
    for i, s in enumerate(sents):
        sentences.append(Document(page_content=f"[{d.metadata['title']}] {s}",
                                  metadata={**d.metadata, "sent_idx": i, "chunk_id": f"{d.metadata['reg_id']}-s{i}"}))
print("문장 수:", len(sentences))
sent_store = memory_store(sentences)

def sentence_window_search(q, k, window=2):
    out = []
    for hit in sent_store.similarity_search(q, k=k):
        sents, i = sent_by_reg[hit.metadata["reg_id"]], hit.metadata["sent_idx"]
        text = " ".join(sents[max(0, i - window): i + window + 1])          # 앞뒤 2문장 붙이기
        out.append(Document(page_content=f"[{hit.metadata['title']}] {text}", metadata=hit.metadata))
    return out

results_B["Sentence Window"] = eval_retrieval(sentence_window_search, 4)

문장 수: 1669


Batches: 100%|██████████| 53/53 [00:26<00:00,  2.02it/s]


## B-4. 가상 질문 인덱스 검색 (HyQE) ⏱
- 청크마다 LLM이 **"이 청크로 답할 수 있는 질문" 2개**를 미리 만들어 저장
- 검색할 때는 사용자 질문과 **가상 질문끼리** 비교 (질문 ↔ 질문) → 질문과 문서의 **문체 차이(의미 간극)** 를 줄임
- 매칭된 가상 질문이 가리키는 **원본 청크**를 LLM에 전달
- `batch(…, max_concurrency=8)`: LLM 호출을 8개씩 **동시에** 보내서 시간을 줄임

In [28]:
class HypoQuestions(BaseModel):
    questions: list[str] = Field(description="이 청크의 내용으로 답할 수 있는 한국어 질문 2개")

qgen = llm.with_structured_output(HypoQuestions, method="function_calling")
prompts = [f"다음 대학 규정 청크를 읽고, 학생이나 교직원이 이 내용으로 답을 얻을 수 있는 질문 2개를 만들어라.\n\n{c.page_content}"
           for c in sub_chunks]
outs = qgen.batch(prompts, config={"max_concurrency": 8}, return_exceptions=True)   # 실패한 호출은 건너뜀

hyqe_docs = []
for c, o in zip(sub_chunks, outs):
    if isinstance(o, Exception) or o is None:
        continue
    for j, q in enumerate(o.questions[:2]):
        hyqe_docs.append(Document(page_content=q, metadata={"parent": c.metadata["chunk_id"],
                                                            "chunk_id": f"{c.metadata['chunk_id']}-q{j}"}))
print("가상 질문 수:", len(hyqe_docs))
print("예시:", sub_chunks[0].page_content[:60], "→", [d.page_content for d in hyqe_docs[:2]])
hyqe_store = memory_store(hyqe_docs)

def question_index_search(store):
    """질문 인덱스 → 원본 청크 k개 (HyQE·Atomic 공통)"""
    def search(q, k):
        out, seen = [], set()
        for hit in store.similarity_search(q, k=k * 4):
            cid = hit.metadata["parent"]
            if cid not in seen:
                seen.add(cid)
                out.append(chunk_by_id[cid])
            if len(out) == k:
                break
        return out
    return search

results_B["HyQE"] = eval_retrieval(question_index_search(hyqe_store), 4)

I1024 19:14:57.925241     266 chttp2_transport.cc:1400] ipv4:127.0.0.1:43033: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11}
E1024 19:14:57.925347     266 chttp2_transport.cc:1432] ipv4:127.0.0.1:43033: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms


가상 질문 수: 410
예시: [명지대학교 학칙] 명지대학교 학칙 제 1 장 총 칙

제1조(목적) 이 학칙은 명지대학교(이하 "우리 대학 → ['명지대학교 학칙 제1조의 목적은 무엇인가?', '명지대학교의 교육목적을 설정한 근거는 무엇인가?']


Batches: 100%|██████████| 13/13 [00:00<00:00, 15.78it/s]


## B-5. 가상 문서 인덱스 검색 (HyDE)
- HyQE와 **반대 방향**: 질문을 받으면 LLM이 먼저 **그럴듯한 가상 답변(규정 조문 모양)** 을 씀
- 그 가상 답변으로 검색 → 짧은 질문보다 실제 조문과 **모양이 비슷해서** 더 잘 찾을 수 있음
- 가상 답변의 내용이 틀려도 괜찮음 (검색용으로만 쓰고, 답변은 실제 청크로 만듦)

In [29]:
from functools import lru_cache

HYDE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "너는 대학 규정 작성자다. 질문에 답하는 규정 조문을 '제N조(제목) …' 형식으로 3문장 이내로 작성하라. "
               "정확한 숫자를 몰라도 그럴듯하게 작성하라."),
    ("human", "{question}")])
hyde_chain = HYDE_PROMPT | llm | StrOutputParser()

@lru_cache(maxsize=None)                     # 같은 질문은 한 번만 LLM 호출 (저장해 두고 재사용)
def hyde_vector(q):
    return tuple(embeddings.embed_query(hyde_chain.invoke({"question": q})))

def hyde_search(q, k):
    return sub_store.similarity_search_by_vector(list(hyde_vector(q)), k=k)

print("[가상 답변 예시]", hyde_chain.invoke({"question": golden[11]["question"]}))
results_B["HyDE"] = eval_retrieval(hyde_search, 4)

[가상 답변 예시] 제5조(도서 대출) ① 학부생은 도서관에서 단행본 5권까지, 최대 14일 동안 대출할 수 있다. ② 대출 기간은 도서관 사정에 따라 연장 또는 조정될 수 있으며, 반납 예정일이 지난 경우 연체료가 부과된다.


## B-6. Atomic 청킹을 활용한 Question 기반 검색 ⏱
- 문제: 청크 임베딩은 청크 안 정보들의 **평균값** → 한 청크에 서로 다른 정보가 섞여 있으면 특정 정보를 정확히 못 찾음
- 해결: 청크를 **원자(Atomic) 단위 사실**(한 문장 = 한 사실)로 나누고, 사실마다 **합성 질문**을 만들어 저장
- 검색은 합성 질문 기준 / LLM에는 연관된 **원본 청크**(풍부한 문맥) 전달

In [30]:
class AtomicFacts(BaseModel):
    facts: list[str] = Field(description="청크를 하나의 사실만 담은 짧은 문장으로 나눈 목록 (최대 5개)")
    questions: list[str] = Field(description="각 사실에 1:1로 대응하는 질문 (facts와 같은 순서, 같은 개수)")

atomizer = llm.with_structured_output(AtomicFacts, method="function_calling")
prompts = [f"다음 대학 규정 청크를 원자 단위 사실(숫자·기간·대상 포함) 최대 5개로 나누고, 사실마다 그 사실을 묻는 질문을 만들어라.\n\n{c.page_content}"
           for c in sub_chunks]
outs = atomizer.batch(prompts, config={"max_concurrency": 8}, return_exceptions=True)

atomic_docs = []
for c, o in zip(sub_chunks, outs):
    if isinstance(o, Exception) or o is None:
        continue
    for j, (fact, q) in enumerate(zip(o.facts, o.questions)):
        atomic_docs.append(Document(page_content=f"{q} {fact}",            # 합성 질문 + 원자 사실
                                    metadata={"parent": c.metadata["chunk_id"], "chunk_id": f"{c.metadata['chunk_id']}-a{j}"}))
print("원자 단위 수:", len(atomic_docs))
print("예시:", [d.page_content for d in atomic_docs[:3]])
atomic_store = memory_store(atomic_docs)

results_B["Atomic 청킹"] = eval_retrieval(question_index_search(atomic_store), 4)

원자 단위 수: 764
예시: ['명지대학교 학칙 제1조의 목적은 무엇인가? 명지대학교 학칙 제1조는 교육목적 설정 및 교육조직, 학사운영, 교육과정 등에 관한 사항을 규정함을 목적으로 한다.', '명지대학교 학칙 제2조에 명시된 설립정신은 무엇인가? 명지대학교 학칙 제2조는 학교법인 명지학원의 설립정신인 기독교 정신을 바탕으로 학술의 심오한 이론과 올바른 활용방법을 연구·교수하는 것을 규정한다.', '명지대학교 학칙 제2조의 교육목적은 무엇인가? 명지대학교 학칙 제2조는 국가발전과 민족문화 창달을 통해 인류사회 평화와 발전에 기여 봉사할 인재양성을 교육목적으로 한다.']


Batches: 100%|██████████| 24/24 [00:03<00:00,  7.91it/s]


## B-7. Retrieval 전략 종합 비교
- 높을수록 좋음: hit · precision · recall · mrr · ndcg · ctx_precision / 낮을수록 좋음: ctx_chars(LLM에 넘기는 글자 수) · search_ms(검색 시간)
- **보는 법**: 정확도가 비슷하면 `ctx_chars`가 작은 쪽이 비용·속도에 유리. Parent-Child는 정확도가 올라도 글자 수가 큼

In [31]:
table_B = compare(results_B)
table_B

,hit,precision,recall,mrr,ndcg,ctx_precision,ctx_chars,search_ms
기본 청킹,1.000,0.272,0.978,0.833,0.862,0.826,1658.957,46.749
계층 인덱스,0.870,0.228,0.826,0.754,0.753,0.746,1653.870,39.512
Parent-Child,0.913,0.250,0.913,0.739,0.781,0.732,3539.652,100.784
Sentence Window,0.957,0.435,0.957,0.822,0.862,0.810,1311.261,134.906
HyQE,0.522,0.130,0.478,0.457,0.452,0.457,1596.217,39.841
HyDE,1.000,0.272,0.978,0.797,0.837,0.790,1622.435,1406.225
Atomic 청킹,0.522,0.141,0.478,0.522,0.488,0.522,1614.609,69.339


In [32]:
# 문항별로 어떤 전략이 정답을 1위로 올렸는지 (MRR) 비교
pd.DataFrame({name: df.set_index("id")["mrr"] for name, df in results_B.items()}).round(2)

,기본 청킹,계층 인덱스,Parent-Child,Sentence Window,HyQE,HyDE,Atomic 청킹
id,,,,,,,
q01,1.00,0.00,1.00,1.00,1.0,1.00,1.0
q02,0.50,0.50,0.33,0.33,0.0,0.50,1.0
q03,0.50,1.00,0.50,0.50,1.0,0.33,1.0
q04,1.00,0.00,1.00,1.00,0.5,0.33,1.0
q05,1.00,1.00,0.00,0.00,1.0,1.00,1.0
q06,0.50,0.00,0.50,0.25,1.0,0.50,1.0
q07,0.50,0.50,0.50,1.00,1.0,1.00,1.0
q08,0.33,0.50,0.33,1.00,0.0,0.50,0.0
q09,1.00,1.00,1.00,1.00,1.0,1.00,0.0


---
# Part C. Search 전략

## C-1. Sparse / Dense 벡터 비교
- **Sparse(희소) 벡터**: 단어(여기서는 2글자 조각)마다 칸이 하나 → 칸은 수만 개, 대부분 0
- **Dense(밀집) 벡터**: 1024칸 모두 값이 있음 → 뜻을 압축해서 표현
- 한국어는 띄어쓰기 단위가 길어서 **2글자씩 겹쳐 자르기**(bigram)로 단어를 나눔 (예: `학사경고` → `학사` `사경` `경고`)

In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer

def ko_tokenize(text):
    words = re.findall(r"[가-힣A-Za-z0-9]+", text)
    return [w[i:i + 2] for w in words for i in range(max(1, len(w) - 1))]     # 2글자씩 겹쳐 자르기

print("토큰 예시:", ko_tokenize("학사경고를 받는 성적 기준"))

tfidf = TfidfVectorizer(analyzer=ko_tokenize).fit([c.page_content for c in chunks_clean])
sentence = "학부생은 도서관에서 책을 몇 권 빌릴 수 있나요?"
sv = tfidf.transform([sentence])
dv = np.array(embeddings.embed_query(sentence))
print(f"Sparse: 전체 {sv.shape[1]:,}칸 중 값이 있는 칸 {sv.nnz}개")
print(f"Dense : 전체 {len(dv)}칸 중 값이 있는 칸 {int((np.abs(dv) > 1e-6).sum())}개")

토큰 예시: ['학사', '사경', '경고', '고를', '받는', '성적', '기준']
Sparse: 전체 20,681칸 중 값이 있는 칸 11개
Dense : 전체 1024칸 중 값이 있는 칸 1023개


## C-2. Hybrid Search — 키워드(BM25) + 벡터 검색 결합
- **BM25**: 질문의 단어가 그대로 들어 있는 청크를 찾음 → 숫자·전문용어·고유명사(예: '제적', 'A0')에 강함
- **벡터 검색**: 단어가 달라도 뜻이 비슷하면 찾음 → 바꿔 말한 질문에 강함
- **RRF(Reciprocal Rank Fusion)**: 두 검색의 **순위**를 합치는 방법. 점수 = Σ 1 / (60 + 순위)
  - 두 검색 모두에서 상위에 오른 청크가 최종 1위가 됨

In [34]:
from langchain_community.retrievers import BM25Retriever

bm25 = BM25Retriever.from_documents(chunks_clean, preprocess_func=ko_tokenize)

def bm25_search(q, k):
    bm25.k = k
    return bm25.invoke(q)

def rrf_fuse(result_lists, k, c=60):
    """여러 검색 결과 목록 → RRF 점수로 합쳐서 상위 k개"""
    scores, doc_by_id = {}, {}
    for results in result_lists:
        for rank, d in enumerate(results, start=1):
            cid = d.metadata["chunk_id"]
            scores[cid] = scores.get(cid, 0) + 1 / (c + rank)
            doc_by_id[cid] = d
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [doc_by_id[cid] for cid in ranked[:k]]

def hybrid_search(q, k, n=20):
    return rrf_fuse([dense_search(q, n), bm25_search(q, n)], k)    # 각각 20개 찾아서 합침

q = "학생 징계의 종류에는 무엇이 있나요?"
for name, fn in [("BM25", bm25_search), ("Dense", dense_search), ("Hybrid", hybrid_search)]:
    print(f"[{name}]", [f"{d.metadata['reg_id']} {d.metadata['article']}" for d in fn(q, 4)])

results_C = {"Dense (부칙 제거)": results_A["부칙 제거"],
             "BM25": eval_retrieval(bm25_search, 4),
             "Hybrid (RRF)": eval_retrieval(hybrid_search, 4)}
compare(results_C)

[BM25] ['1-1 제60조', '5-2-1 제57조', '1-1 제88조', '1-2 제25조']
[Dense] ['5-2-1 제57조', '3-2-3 제11조', '3-2-3 제12조', '3-2-3 제7조']
[Hybrid] ['5-2-1 제57조', '1-1 제60조', '3-2-3 제12조', '1-1 제88조']


,hit,precision,recall,mrr,ndcg,ctx_precision,ctx_chars,search_ms
Dense (부칙 제거),0.870,0.239,0.870,0.728,0.758,0.717,1696.304,333.049
BM25,0.696,0.185,0.674,0.641,0.638,0.641,1645.696,12.690
Hybrid (RRF),0.913,0.250,0.891,0.696,0.736,0.699,1671.609,356.289


## C-3. Metadata Filtering — Pre-Filter vs Post-Filter
- **Pre-Filter**: 메타데이터 조건으로 **먼저 거른 뒤** 그 안에서 벡터 검색 → 항상 k개를 채움
  - Milvus에서는 `expr` 조건식으로 지정 (예: `reg_id in ["2-1-1", "2-2-1"]`)
- **Post-Filter**: 벡터 검색으로 **먼저 k개를 찾은 뒤** 조건으로 거름 → 조건에 맞는 것이 적으면 **결과가 k개보다 적어짐(결과 붕괴)**
- 예: '휴학' 질문을 **학부 규정(학칙·학사과정 시행규칙)** 으로만 한정

In [35]:
q = "휴학은 최대 몇 학기까지 할 수 있나요?"
UNDERGRAD = ["2-1-1", "2-2-1"]                                         # 학칙, 학칙 시행규칙(학사과정)

pre = store_clean.similarity_search(q, k=4, expr=f"reg_id in {json.dumps(UNDERGRAD)}")   # Pre-Filter
post = [d for d in store_clean.similarity_search(q, k=4) if d.metadata["reg_id"] in UNDERGRAD]   # Post-Filter

print("[필터 없음]", [d.metadata["title"] for d in store_clean.similarity_search(q, k=4)])
print(f"[Pre-Filter ] {len(pre)}개:", [d.metadata["title"] for d in pre])
print(f"[Post-Filter] {len(post)}개:", [d.metadata["title"] for d in post], "← k=4보다 적으면 '결과 붕괴'")

[필터 없음] ['학칙 시행규칙(학사과정)', '명지대학교 학칙', '미래교육원 규정', '학칙 시행규칙(학사과정)']
[Pre-Filter ] 4개: ['학칙 시행규칙(학사과정)', '명지대학교 학칙', '학칙 시행규칙(학사과정)', '학칙 시행규칙(학사과정)']
[Post-Filter] 3개: ['학칙 시행규칙(학사과정)', '명지대학교 학칙', '학칙 시행규칙(학사과정)'] ← k=4보다 적으면 '결과 붕괴'


In [36]:
# 편(part) 조건으로 Pre-Filter: 같은 질문이라도 검색 범위를 좁히면 결과가 달라짐
for part in ["제2편 학칙,시행규칙", "제5편 부속,부설교육,연구소,위원회"]:
    res = store_clean.similarity_search(q, k=3, expr=f'part == "{part}"')
    print(f"[{part}]", [f"{d.metadata['title']} {d.metadata['article']}" for d in res])

[제2편 학칙,시행규칙] ['학칙 시행규칙(학사과정) 제16조', '명지대학교 학칙 제19조', '학칙 시행규칙(학사과정) 제18조']
[제5편 부속,부설교육,연구소,위원회] ['미래교육원 규정 제16조', '한국기록관리학교육원 규정 제14조', '미래교육원 규정 제6조']


---
# Part D. 검색 후 절차 (Post-Retrieval)

## D-1. ReRank — 넉넉히 찾고, 정밀하게 다시 줄 세우기
- 1차: Hybrid로 후보 **20개**를 빠르게 찾음
- 2차: **Cross-Encoder 리랭커**(`BAAI/bge-reranker-v2-m3`)가 (질문, 청크)를 **함께 읽고** 관련도를 다시 매김 → 상위 4개만 LLM에 전달
- 임베딩 검색은 질문과 청크를 **따로** 숫자로 바꿔 비교 → 빠르지만 덜 정밀 / 리랭커는 **같이** 읽음 → 느리지만 정밀

In [37]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512)     # 처음 실행 시 모델 다운로드

def rerank(q, docs, k):
    scores = reranker.predict([(q, d.page_content) for d in docs], batch_size=16)
    order = np.argsort(scores)[::-1][:k]                               # 점수 높은 순
    return [docs[i] for i in order]

def hybrid_rerank_search(q, k, n=20):
    return rerank(q, hybrid_search(q, n), k)

results_C["Hybrid + ReRank"] = eval_retrieval(hybrid_rerank_search, 4)
compare(results_C)

,hit,precision,recall,mrr,ndcg,ctx_precision,ctx_chars,search_ms
Dense (부칙 제거),0.870,0.239,0.870,0.728,0.758,0.717,1696.304,333.049
BM25,0.696,0.185,0.674,0.641,0.638,0.641,1645.696,12.690
Hybrid (RRF),0.913,0.250,0.891,0.696,0.736,0.699,1671.609,356.289
Hybrid + ReRank,1.000,0.283,1.000,0.877,0.908,0.873,1750.000,639.971


## D-2. 프롬프트 압축 — 질문과 관련된 문장만 남기기
- 청크를 문장으로 나누고, 질문과 **유사도가 높은 문장 절반만** 남김 → LLM에 넘기는 글자 수 감소, 정보 밀도 향상
- 위험: 정답 문장까지 지울 수 있음 → 검색 지표(Recall)로 확인

In [38]:
def compress_docs(q, docs, keep=0.5):
    qv = np.array(embeddings.embed_query(q))
    out = []
    for d in docs:
        sents = split_sentences(d.page_content)
        if len(sents) <= 2:                                            # 짧은 청크는 그대로
            out.append(d)
            continue
        scores = np.array(embeddings.embed_documents(sents)) @ qv
        top = sorted(np.argsort(scores)[::-1][:max(1, int(len(sents) * keep))])   # 원래 순서 유지
        out.append(Document(page_content=" ".join(sents[i] for i in top), metadata=d.metadata))
    return out

def compressed_search(q, k):
    return compress_docs(q, hybrid_rerank_search(q, k))

results_C["Hybrid + ReRank + 압축"] = eval_retrieval(compressed_search, 4)
compare(results_C)

,hit,precision,recall,mrr,ndcg,ctx_precision,ctx_chars,search_ms
Dense (부칙 제거),0.870,0.239,0.870,0.728,0.758,0.717,1696.304,333.049
BM25,0.696,0.185,0.674,0.641,0.638,0.641,1645.696,12.690
Hybrid (RRF),0.913,0.250,0.891,0.696,0.736,0.699,1671.609,356.289
Hybrid + ReRank,1.000,0.283,1.000,0.877,0.908,0.873,1750.000,639.971
Hybrid + ReRank + 압축,1.000,0.283,1.000,0.877,0.908,0.873,978.652,747.831


## D-3. 질의 변환 — StepBack(역추적) & 서브쿼리
- **StepBack**: 한 걸음 물러나 **더 큰 그림의 질문**을 만들어 함께 검색 (예: '최소 몇 학기?' → '졸업 요건과 학기당 이수 학점 기준은?')
- **서브쿼리**: 복잡한 질문을 **하위 질문 여러 개**로 나눠 각각 검색 → 결과를 모아서 답변
- 대상: 근거가 두 곳에 나뉜 **종합형 3문항**(q21·q22·q23)

In [39]:
class SubQuestions(BaseModel):
    questions: list[str] = Field(description="원래 질문에 답하는 데 필요한 하위 질문 2~3개")

decomposer = llm.with_structured_output(SubQuestions, method="function_calling")
STEPBACK_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "사용자의 구체적인 질문을 한 단계 일반화한 '배경 질문' 한 문장으로 바꿔라. 질문만 출력하라."),
    ("human", "{question}")])
stepback_chain = STEPBACK_PROMPT | llm | StrOutputParser()

def merge_unique(result_lists, limit):
    """여러 검색 결과를 번갈아 가며 중복 없이 합치기"""
    out, seen = [], set()
    for rank in range(max(len(r) for r in result_lists)):
        for r in result_lists:
            if rank < len(r) and r[rank].metadata["chunk_id"] not in seen:
                seen.add(r[rank].metadata["chunk_id"])
                out.append(r[rank])
    return out[:limit]

def stepback_pipeline(question):
    t0 = time.time()
    general = stepback_chain.invoke({"question": question})
    docs = merge_unique([hybrid_rerank_search(question, 3), hybrid_rerank_search(general, 3)], 6)
    answer = answer_chain.invoke({"context": format_docs(docs), "question": question})
    return {"answer": answer, "docs": docs, "total_ms": (time.time() - t0) * 1000, "llm_calls": 2, "extra": general}

def subquery_pipeline(question):
    t0 = time.time()
    subs = decomposer.invoke(f"다음 질문에 답하려면 규정에서 각각 무엇을 찾아봐야 하는지 하위 질문 2~3개로 나눠라.\n질문: {question}").questions
    docs = merge_unique([hybrid_rerank_search(s, 3) for s in [question] + subs[:3]], 6)
    answer = answer_chain.invoke({"context": format_docs(docs), "question": question})
    return {"answer": answer, "docs": docs, "total_ms": (time.time() - t0) * 1000, "llm_calls": 2, "extra": subs}

q23 = next(g for g in golden if g["id"] == "q23")
for name, fn in [("기본(Hybrid+ReRank)", make_rag(hybrid_rerank_search)), ("StepBack", stepback_pipeline), ("서브쿼리", subquery_pipeline)]:
    out = fn(q23["question"])
    print(f"===== {name} =====")
    if "extra" in out:
        print("변환된 질문:", out["extra"])
    print("근거 이름표:", label_docs(out["docs"], q23))
    print(out["answer"], "\n")

===== 기본(Hybrid+ReRank) =====
근거 이름표: ['무관', '무관', '근거2', '근거1']
인공지능·소프트웨어융합대학 학생의 매 학기 최대 수강학점은 **18학점**입니다.

졸업학점은 **134학점**입니다.

따라서 최소 학기 수는 다음과 같이 계산합니다.

134 ÷ 18 = 7.444...

학기 수는 정수이므로, 7학기만으로는 126학점에 그쳐 134학점에 부족합니다. 따라서 최소 **8학기**가 필요합니다.

**답: 8학기**

(출처: 학칙 제31조 제1항 제2호, 학칙 시행규칙 제108조 제1항 제1호) 

===== StepBack =====
변환된 질문: 인공지능·소프트웨어융합대학 학생이 매 학기 최대 학점만 수강하면 졸업까지 최소 몇 학기가 필요한가요?
근거 이름표: ['무관', '무관', '근거2']
인공지능·소프트웨어융합대학 학생의 경우, 학칙 시행규칙(학사과정) 제32조 제1항에 따라 매 학기 수강신청 기준학점은 18학점입니다. 또한 같은 조 제2항에 따라 직전학기와 전학년 평균평점이 4.0 이상인 학생은 3학점을 추가하여 최대 21학점까지 신청할 수 있습니다.

졸업학점은 같은 시행규칙 제108조 제1항 제1호에 따라 인공지능·소프트웨어융합대학은 134학점입니다.

따라서 최대 학점만 수강한다고 가정하면, 134학점을 21학점으로 나누어 계산합니다.

134 ÷ 21 = 6.3809...

즉, 6학기 동안 21학점씩 수강하면 126학점을 이수하게 되고, 나머지 8학점은 7학기에 추가로 이수해야 합니다.

따라서 졸업까지 최소 **7학기**가 필요합니다.

(출처: 학칙 시행규칙(학사과정) 제32조 제1항·제2항, 제108조 제1항 제1호) 

===== 서브쿼리 =====
변환된 질문: ['인공지능·소프트웨어융합대학 학생의 졸업 이수 학점은 규정에서 몇 학점인지 확인해야 합니다.', '매 학기 최대 수강 가능 학점은 규정에서 몇 학점인지 확인해야 합니다.', '졸업까지 필요한 최소 학기 수는 졸업 이수 학

In [40]:
# 종합형 3문항 채점 ⏱ (질문당 LLM 답변 + 채점)
multi = [g for g in golden if g["type"] == "종합형"]
res_multi = {name: evaluate(fn, items=multi, relevancy=False)
             for name, fn in [("Hybrid+ReRank", make_rag(hybrid_rerank_search)),
                              ("StepBack", stepback_pipeline), ("서브쿼리", subquery_pipeline)]}
pd.DataFrame({name: df.set_index("id")["correct"] for name, df in res_multi.items()})

100%|██████████| 3/3 [00:28<00:00,  9.45s/it]


,Hybrid+ReRank,StepBack,서브쿼리
id,,,
q21,True,True,True
q22,True,False,True
q23,True,True,True


---
# Part E. Collection 분리 / 분산

## E-1. 편(Part)별 Collection 분리 + 질문 라우팅 ⏱ (임베딩 약 3분)
- 강의자료 3단계: ① 데이터 접근 패턴 분석 → ② 분할 기준 수립 → ③ 각 Collection 크기 축소로 성능·확장성 확보
- 규정집은 **편(제1편~제6편, 부록)** 단위로 성격이 뚜렷함 → 편마다 Collection을 따로 만듦
- 질문이 들어오면 LLM **라우터**가 어느 편에서 찾을지 골라서 그 Collection만 검색

In [41]:
PART_GUIDE = {
    "제1편 명지학원": "학교법인 정관, 법인 인사·보수·표창 규정",
    "제2편 학칙,시행규칙": "학칙과 학사·대학원 시행규칙: 수업연한, 학점, 수강, 휴학, 성적, 졸업, 학사경고",
    "제3편 학사행정": "학사처리 내규, 융합전공, 현장실습, 명예졸업, 학생 활동·집회, 학생 징계, 각종 사업단",
    "제4편 일반행정": "직제, 교원·직원 인사, 연구년, 복무, 여비, 시설·안전·교통(전동킥보드 등)",
    "제5편 부속,부설교육,연구소,위원회": "도서관, 언어교육원, 상담센터, 박물관, 신문사·방송국, 연구소, 위원회",
    "제6편 산학협력단": "산학협력단 정관, 지식재산권, 연구비·연구 관리",
    "부록,참고규정": "교직원친화회, 총동문회 회칙",
}
PARTS = list(PART_GUIDE)

part_stores = {}
for i, part in enumerate(PARTS):
    part_chunks = [c for c in chunks_clean if c.metadata["part"] == part]
    part_stores[part] = build_store(part_chunks, f"mju_part_{i}")
    print(f"{part}: {len(part_chunks)}개 청크")

제1편 명지학원: 366개 청크
제2편 학칙,시행규칙: 429개 청크
제3편 학사행정: 859개 청크
제4편 일반행정: 1470개 청크
제5편 부속,부설교육,연구소,위원회: 395개 청크
제6편 산학협력단: 168개 청크
부록,참고규정: 21개 청크


In [42]:
class Route(BaseModel):
    parts: list[Literal[tuple(PARTS)]] = Field(description="질문의 답이 있을 가능성이 높은 편 (1~2개, 가능성 높은 순)")

router = llm.with_structured_output(Route, method="function_calling")
guide = "\n".join(f"- {p}: {desc}" for p, desc in PART_GUIDE.items())

@lru_cache(maxsize=None)
def route(q):
    parts = router.invoke(f"[규정집 편 안내]\n{guide}\n\n[질문]\n{q}\n\n이 질문의 답을 찾을 편을 골라라.").parts
    return tuple(parts[:2]) or ("제2편 학칙,시행규칙",)

def routed_search(q, k):
    hits = []
    for part in route(q):                                              # 고른 편의 Collection만 검색
        hits += part_stores[part].similarity_search_with_score(q, k=k)
    hits.sort(key=lambda x: -x[1])                                     # 유사도 높은 순으로 합치기
    return [d for d, _ in hits[:k]]

# 라우팅 정확도: 정답 근거가 있는 편을 골랐나?
part_of = {d.metadata["reg_id"]: d.metadata["part"] for d in docs}
ok = [all(part_of[src] in route(g["question"]) for src, _ in g["evidence"]) for g in golden if g["evidence"]]
print(f"라우팅 정확도: {np.mean(ok):.0%}")

results_E = {"단일 Collection": results_A["부칙 제거"], "편별 분리 + 라우팅": eval_retrieval(routed_search, 4)}
compare(results_E)

라우팅 정확도: 100%


,hit,precision,recall,mrr,ndcg,ctx_precision,ctx_chars,search_ms
단일 Collection,0.870,0.239,0.870,0.728,0.758,0.717,1696.304,333.049
편별 분리 + 라우팅,0.913,0.250,0.913,0.783,0.810,0.772,1678.696,124.091


> **보는 법**: 검색 범위가 좁아져 노이즈가 줄면 MRR이 오르지만, 라우터가 편을 잘못 고르면 **아예 못 찾음(Hit=0)** → 라우팅 정확도를 꼭 함께 확인
> `search_ms`에는 라우터 LLM 호출 시간은 빠져 있음 (처음 한 번만 호출 후 저장)

## E-2. 샤딩(Sharding) 설정 — 참고
- 데이터가 아주 많아지면 하나의 Collection을 여러 **샤드(조각)** 로 나눠 여러 서버(노드)에 분산 저장
- 강의자료의 Weaviate 스키마 예시와 같은 의미를 Milvus에서는 아래처럼 지정 (Milvus Lite는 단일 파일이라 실제 분산은 안 됨 → 개념 확인용)

```python
# Weaviate (강의자료 예시)
{"class": "Users", "vectorizer": "text2vec-transformers",
 "shardingConfig": {"strategy": "hash", "desiredCount": 4, "key": "user_id"},
 "replicationConfig": {"factor": 2}}

# Milvus (서버 버전)
client.create_collection(collection_name="mju_rules", schema=schema,
                         num_shards=4)                     # 샤드 4개로 분산 저장
# 레플리카: 컬렉션 로드 시 replica_number=2 지정
```

---
# 종합 — 기본 RAG vs Advanced RAG (답변까지 채점) ⏱

- 같은 26문항으로 **답변을 만들고 LLM이 채점**: 3장 기본 RAG vs Advanced RAG(부칙 제거 + Hybrid + ReRank)
- 시간 절약을 위해 Answer Relevancy는 생략 (Faithfulness·Correctness 중심)

In [43]:
final = {"기본 RAG (3장)": evaluate(make_rag(dense_fn(store_base)), relevancy=False),
         "Advanced RAG": evaluate(make_rag(hybrid_rerank_search), relevancy=False)}
pd.DataFrame({name: summary(df) for name, df in final.items()}).round(3)

100%|██████████| 26/26 [02:01<00:00,  4.69s/it]


,기본 RAG (3장),Advanced RAG
hit,0.870,1.000
precision,0.239,0.283
recall,0.870,1.000
mrr,0.728,0.877
ndcg,0.758,0.908
ctx_precision,0.717,0.873
faithfulness,1.000,1.000
correct,0.885,0.962
ctx_chars,1748.615,1762.077
total_ms,1902.080,2421.599


In [44]:
# 검색 지표 전체 요약 (전체 규정, K=4)
compare({**results_A, **{k: v for k, v in results_C.items() if k != "Dense (부칙 제거)"}, "편별 분리 + 라우팅": results_E["편별 분리 + 라우팅"]})

,hit,precision,recall,mrr,ndcg,ctx_precision,ctx_chars,search_ms
3장 기본,0.870,0.239,0.870,0.728,0.758,0.717,1717.609,390.690
부칙 제거,0.870,0.239,0.870,0.728,0.758,0.717,1696.304,333.049
BM25,0.696,0.185,0.674,0.641,0.638,0.641,1645.696,12.690
Hybrid (RRF),0.913,0.250,0.891,0.696,0.736,0.699,1671.609,356.289
Hybrid + ReRank,1.000,0.283,1.000,0.877,0.908,0.873,1750.000,639.971
Hybrid + ReRank + 압축,1.000,0.283,1.000,0.877,0.908,0.873,978.652,747.831
편별 분리 + 라우팅,0.913,0.250,0.913,0.783,0.810,0.772,1678.696,124.091


I1024 19:27:40.615887     266 chttp2_transport.cc:1400] ipv4:127.0.0.1:43033: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11}
E1024 19:27:40.615975     266 chttp2_transport.cc:1432] ipv4:127.0.0.1:43033: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 20000ms


---
## 정리 & 직접 해보기

### 오늘 배운 것
1. **검색 전 절차**: 데이터를 줄이고(부칙 제거) 다듬고(LLM 정련) 묶으면(중복 그룹화) 검색할 대상 자체가 좋아짐
2. **Retrieval 전략**: '무엇으로 검색하고 무엇을 넘길지'를 분리 (Small to Big, HyQE, Atomic) → 정확도와 글자 수(비용)를 함께 비교
3. **Search 전략**: 키워드(BM25)와 벡터를 합치면(Hybrid) 서로의 약점을 보완 / 메타데이터 Pre-Filter는 결과 붕괴가 없음
4. **검색 후 절차**: ReRank로 순위를 다듬고, 서브쿼리로 근거가 나뉜 질문을 해결
5. 모든 개선은 **같은 문제지로 전후 점수를 비교**해서 효과를 확인

### 직접 해보기 과제
1. `hybrid_search`의 후보 수 `n`을 10·40으로 바꿔 Hybrid + ReRank 점수와 검색 시간 비교
2. Parent-Child의 자식 크기(200)·부모 크기(1000)를 바꿔 `ctx_chars`와 정확도의 균형 찾기
3. `PART_GUIDE` 설명을 고쳐 라우팅 정확도를 100%로 만들어 보기